In [5]:
lo_map = { 
}

import numpy as np
all_matrices = []
for mask in range(2**16):
    bits = [(mask >> i) & 1 for i in range(15, -1, -1)]
    
    matrix = np.array(bits).reshape(4, 4)
    all_matrices.append(matrix)
all_matrices = np.array(all_matrices)
for i in range(2**16):
    lo_map[''.join(all_matrices[i].flatten().astype(str))]={}

for state in range(0,2**16):
    transitions={}
    for i in range(4):
        for j in range(4):
            transitions[(i*4)+j]=[]
            currentarray=all_matrices[state].copy()
            if np.array_equal(currentarray,all_matrices[0]):
                continue
            else:
                currentarray[i][j]=1-currentarray[i][j]
                if(i!=0):
                    currentarray[i-1][j]=1-currentarray[i-1][j]
                if(i!=3):
                    currentarray[i+1][j]=1-currentarray[i+1][j]
                if(j!=0):
                    currentarray[i][j-1]=1-currentarray[i][j-1]
                if(j!=3):
                    currentarray[i][j+1]=1-currentarray[i][j+1]
                
                if np.array_equal(currentarray,all_matrices[0]):
                    transitions[(i*4)+j].append((1,''.join(currentarray.flatten().astype(str)),0,True))
                else:
                    transitions[(i*4)+j].append((1,''.join(currentarray.flatten().astype(str)),-1,False))
    lo_map[''.join(all_matrices[state].flatten().astype(str))]=transitions
        


In [2]:
states = list(lo_map.keys())
terminal_state = '0' * 16
actions = list(range(16))
V = {s: 0.0 for s in states}
V[terminal_state] = 0.0
theta = 1e-4
gamma = 0.9

In [3]:
while True:
    delta = 0
    V_new = V.copy()

    for s in states:
        if s == terminal_state:
            continue

        best = float('-inf')

        for a, transitions in lo_map[s].items():
            p=transitions[0][0]
            s_next=transitions[0][1]
            r=transitions[0][2]
            val = p * (r + gamma * V[s_next])
            best = max(best, val)
        
        V_new[s] = best
        delta = max(delta, abs(V[s] - best))

    V = V_new

    if delta < theta:
        break

In [4]:
policy = {}

for s in states:
    if s == terminal_state:
        policy[s] = None
        continue

    best = float('-inf')
    best_action = None

    for a in actions:
        transitions = lo_map[s][a]
        val = 0

        for p, s_next, r, done in transitions:
            val += p * (r + gamma * V[s_next])

        if val > best:
            best = val
            best_action = a

    policy[s] = best_action
